- 我们使用pytorch提供的相应的数据模块来简洁实现线性回归

In [24]:
# 导入相应的库
import torch
import numpy as np
from d2l import torch as d2l
from torch.utils import data

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000) # 生成1000个样本
print('features:', features[0], '\nlabel:', labels[0])

features: tensor([-1.9366, -0.0530]) 
label: tensor([0.5169])


- 调用框架中现有的API来读取数据

In [25]:
def load_array(data_arrays, batch_size, is_train=True):
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays) #传入的是一个列表，*表示解压
    return data.DataLoader(dataset, batch_size, shuffle=is_train) #每次迭代返回batch_size个随机样本
batch_size = 10
data_iter = load_array((features, labels), batch_size)
next(iter(data_iter))

[tensor([[-1.3463,  1.0703],
         [ 0.9111,  0.2039],
         [-0.7743,  0.4788],
         [ 1.7315,  0.6151],
         [-1.3306,  0.9041],
         [-0.6280,  1.4169],
         [ 0.6674, -0.1797],
         [-0.8882, -0.1039],
         [ 0.4012,  1.2718],
         [-1.8731, -0.1231]]),
 tensor([[-2.1130],
         [ 5.3145],
         [ 1.0088],
         [ 5.5729],
         [-1.5341],
         [-1.8917],
         [ 6.1435],
         [ 2.7800],
         [ 0.6626],
         [ 0.8874]])]

- 使用框架的预定义好的层

In [26]:
from torch import nn # nn是神经网络的缩写，包含了构建神经网络的各种组件
net = nn.Sequential(nn.Linear(2, 1)) #输入维度为2，输出维度为1  可以理解为list or array

- 初始化模型参数

In [27]:
net[0].weight.data.normal_(0, 0.01) #权重参数初始化为均值为0，标准差为0.01的正态分布随机数
net[0].bias.data.fill_(0) #偏置参数初始化为0 截距

tensor([0.])

- 计算均方误差使用的是MSELoss类，也称为$L_2$范数

In [28]:
loss = nn.MSELoss() # 计算均方误差使用的是MSELoss类，也称为$L_2$范数

- 实例化SGD实例

In [29]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03) # 实例化SGD实例
# parameters就是包含所有的参数，包括w和b，lr是学习率

- 训练过程代码与之前的从零开始实现时所做的非常相似

In [30]:
num_epochs = 3
for epoch in range(num_epochs):
    for x,y in data_iter:
        l = loss(net(x),y)
        trainer.zero_grad() # 梯度清零
        l.backward() # 反向传播计算梯度 这里pytorch自动求sum了所有样本的损失
        trainer.step() # 更新参数
    with torch.no_grad(): # 评估模型参数
        l = loss(net(features), labels)
        print(f'epoch {epoch + 1}, loss {l:f}')


epoch 1, loss 0.000301
epoch 2, loss 0.000099
epoch 3, loss 0.000100


- 查看训练结果，判断模型是否学到了正确的参数

In [31]:
print(f'w的估计误差: {true_w - net[0].weight.data.reshape(true_w.shape)}')
print(f'b的估计误差: {true_b - net[0].bias.data}')

w的估计误差: tensor([-0.0002, -0.0004])
b的估计误差: tensor([-0.0004])
